TASK FRAMING 

Cílem tohoto projektu je predikovat inflaci v České republice v následujícím kvartálu (t+1) na základě makroekonomických ukazatelů.

Jednotkou pozorování je kvartál.

Úloha je formulována jako regresní problém, kde cílovou proměnnou je inflace v čase t+1.

Jako hlavní metrika pro vyhodnocení modelu bude použita chyba predikce (např. RMSE).

IMPORT KNIHOVEN

In [22]:
import pandas as pd
import numpy as np

DATA DESCRIPTION

Data byla získána z veřejných zdrojů (Eurostat, OECD).

Dataset obsahuje kvartální data za období 1997–2025.

Použité proměnné:
- inflace v ČR (infl_cz)
- inflace v Německu (infl_de)
- úroková sazba (rate)
- cena ropy (oil)
- růst HDP (gdp_growth)

Chybějící hodnoty byly odstraněny pomocí dropna().
Duplicitní záznamy byly odstraněny.

Cílová proměnná (target) byla vytvořena jako inflace posunutá o jedno období dopředu.

NAČTENÍ CSV

Oil (Brent)
Zdroj: FRED
Frekvence: denní
Datum stažení: 28.4.2026
Transformace: průměr na kvartál

In [23]:
oil = pd.read_csv("data/raw/OIL.csv")


# Přidání date
oil["date"] = pd.to_datetime(oil["observation_date"])
oil = oil.set_index("date")

#přejmenování, vyčištění
oil = oil[["DCOILBRENTEU"]]
oil = oil.rename(columns={"DCOILBRENTEU": "oil"})


print(oil.head())
print(oil.tail())


              oil
date             
1987-05-20  18.63
1987-05-21  18.45
1987-05-22  18.55
1987-05-25  18.60
1987-05-26  18.63
               oil
date              
2026-04-14  118.69
2026-04-15  114.93
2026-04-16  116.63
2026-04-17   98.63
2026-04-20  103.40


In [24]:
#převod na kvartál
oil_q = oil.resample("QE").mean()

#Ceny ropy byly převedeny na kvartální frekvenci pomocí aritmetického průměru.

#Poslední neúplné období bylo odstraněno, aby nedocházelo ke zkreslení modelu neúplnými daty
oil_q = oil_q[oil_q.index < "2026-04-01"]

print(oil_q.head())
print(oil_q.tail())

                  oil
date                 
1987-06-30  18.783103
1987-09-30  19.063030
1987-12-31  17.865538
1988-03-31  15.655323
1988-06-30  16.129841
                  oil
date                 
2025-03-31  75.874603
2025-06-30  68.067049
2025-09-30  69.031231
2025-12-31  63.654219
2026-03-31  80.719841


Interest rate (CZ)
Zdroj: FRED
Datum stažení: 28.4.2026
Frekvence: měsíční
Transformace: průměr na kvartál

In [25]:
rate = pd.read_csv("data/raw/Interest_Rate_CZ.csv")

# Přidání date
rate["date"] = pd.to_datetime(rate["observation_date"])
rate = rate.set_index("date")

#přejmenování, vyčištění
rate = rate[["IR3TIB01CZM156N"]]
rate = rate.rename(columns={"IR3TIB01CZM156N": "rate"})

print(rate.head())
print(rate.tail())

                 rate
date                 
1993-01-01  13.599474
1993-02-01  14.473000
1993-03-01  15.223913
1993-04-01  18.888095
1993-05-01  16.900476
                rate
date                
2025-11-01  3.551667
2025-12-01  3.540000
2026-01-01  3.497500
2026-02-01  3.476316
2026-03-01  3.550000


In [26]:
#převod na kvartál
rate_q = rate.resample("QE").mean()
# Měsíční data byla agregována na kvartální pomocí průměru

#Poslední neúplné období bylo odstraněno, aby nedocházelo ke zkreslení modelu neúplnými daty
rate_q = rate_q[rate_q.index < "2026-04-01"]

print(rate_q.head())
print(rate_q.tail())



                 rate
date                 
1993-03-31  14.432129
1993-06-30  16.947857
1993-09-30  12.197303
1993-12-31   9.020530
1994-03-31   8.931739
                rate
date                
2025-03-31  3.784599
2025-06-30  3.575763
2025-09-30  3.495499
2025-12-31  3.539343
2026-03-31  3.507939


In [27]:
hicp_cz = pd.read_csv("data/raw/HICP_CZ.csv")

# Přidání date
hicp_cz["date"] = pd.to_datetime(hicp_cz["TIME_PERIOD"])
hicp_cz = hicp_cz.set_index("date")

# Výběr hodnoty
hicp_cz = hicp_cz[["OBS_VALUE"]]
hicp_cz = hicp_cz.rename(columns={"OBS_VALUE": "hicp"})

# vypočet inflace
# Inflace je počítána jako meziroční procentní změna indexu HICP (YoY). 

infl_cz = hicp_cz.pct_change(12) * 100
infl_cz = infl_cz.rename(columns={"hicp": "infl_cz"})

print(hicp_cz.head())
print(hicp_cz.tail())

            hicp
date            
1996-01-01  57.0
1996-02-01  57.3
1996-03-01  57.6
1996-04-01  58.1
1996-05-01  58.4
             hicp
date             
2025-08-01  156.5
2025-09-01  155.3
2025-10-01  156.1
2025-11-01  155.5
2025-12-01  155.0


HICP (CZ)
Zdroj: Eurostat
Datum stažení: 30.4.2026
Frekvence: měsíční
Transformace:
- výpočet YoY inflace
- převod na kvartál - měsíční data byla agregována na kvartální pomocí průměru

In [28]:
infl_cz_q = infl_cz.resample("QE").mean()

print(infl_cz_q.head())
print(infl_cz_q.tail())

             infl_cz
date                
1996-03-31       NaN
1996-06-30       NaN
1996-09-30       NaN
1996-12-31       NaN
1997-03-31  6.923669
             infl_cz
date                
2024-12-31  3.132911
2025-03-31  2.790329
2025-06-30  2.260827
2025-09-30  2.293271
2025-12-31  1.966685


HICP (DE)
Zdroj: Eurostat
Datum stažení: 30.4.2026
Frekvence: měsíční
Transformace:
- výpočet YoY inflace
- převod na kvartál

In [29]:
hicp_de = pd.read_csv("data/raw/HICP_DE.csv")

# Přidání date
hicp_de["date"] = pd.to_datetime(hicp_de["TIME_PERIOD"])
hicp_de = hicp_de.set_index("date")

# Výběr hodnoty
hicp_de = hicp_de[["OBS_VALUE"]]
hicp_de = hicp_de.rename(columns={"OBS_VALUE": "hicp"})

# vypočet inflace
# Inflace je počítána jako meziroční procentní změna indexu HICP (YoY). 

infl_de = hicp_de.pct_change(12) * 100
infl_de = infl_de.rename(columns={"hicp": "infl_de"})

print(hicp_de.head())
print(hicp_de.tail())

            hicp
date            
1996-01-01  75.1
1996-02-01  75.6
1996-03-01  75.7
1996-04-01  75.6
1996-05-01  75.7
             hicp
date             
2025-08-01  132.5
2025-09-01  132.8
2025-10-01  133.2
2025-11-01  132.6
2025-12-01  132.8


In [30]:
infl_de_q = infl_de.resample("QE").mean()

print(infl_de_q.head())
print(infl_de_q.tail())

             infl_de
date                
1996-03-31       NaN
1996-06-30       NaN
1996-09-30       NaN
1996-12-31       NaN
1997-03-31  1.590829
             infl_de
date                
2024-12-31  2.525796
2025-03-31  2.569026
2025-06-30  2.092048
2025-09-30  2.105469
2025-12-31  2.284426


GDP (CZ)
Zdroj: Eurostat
Datum stažení: 30.4.2026
Frekvence: kvartální

In [31]:
gdp_cz = pd.read_csv("data/raw/GDP_CZ.csv")

# Přidání date
gdp_cz["date"] = pd.PeriodIndex(gdp_cz["TIME_PERIOD"], freq="Q").to_timestamp(how="end")
gdp_cz = gdp_cz.set_index("date")

# výběr hodnot
gdp_cz = gdp_cz[["OBS_VALUE"]]
gdp_cz = gdp_cz.rename(columns={"OBS_VALUE": "gdp"})

print(gdp_cz.head())
print(gdp_cz.tail())

                                  gdp
date                                 
1996-03-31 23:59:59.999999999  60.605
1996-06-30 23:59:59.999999999  65.502
1996-09-30 23:59:59.999999999  66.301
1996-12-31 23:59:59.999999999  66.546
1997-03-31 23:59:59.999999999  60.773
                                   gdp
date                                  
2024-12-31 23:59:59.999999999  119.281
2025-03-31 23:59:59.999999999  120.253
2025-06-30 23:59:59.999999999  120.870
2025-09-30 23:59:59.999999999  121.961
2025-12-31 23:59:59.999999999  122.528


In [32]:
# Growth rate

gdp_cz["gdp_growth"] = gdp_cz["gdp"].pct_change() * 100
gdp_cz = gdp_cz[["gdp_growth"]]

print(gdp_cz.head())
print(gdp_cz.tail())

                               gdp_growth
date                                     
1996-03-31 23:59:59.999999999         NaN
1996-06-30 23:59:59.999999999    8.080191
1996-09-30 23:59:59.999999999    1.219810
1996-12-31 23:59:59.999999999    0.369527
1997-03-31 23:59:59.999999999   -8.675202
                               gdp_growth
date                                     
2024-12-31 23:59:59.999999999    0.588617
2025-03-31 23:59:59.999999999    0.814883
2025-06-30 23:59:59.999999999    0.513085
2025-09-30 23:59:59.999999999    0.902623
2025-12-31 23:59:59.999999999    0.464903


Employment (CZ)
Zdroj: Eurostat
Datum stažení: 30.4.2026
Frekvence: kvartální

In [33]:
emp_cz = pd.read_csv("data/raw/EMP_CZ.csv")

#Filtr
emp_cz = emp_cz[emp_cz["nace_r2"] == "Total - all NACE activities"]

# přidání date
emp_cz["date"] = pd.PeriodIndex(emp_cz["TIME_PERIOD"], freq="Q").to_timestamp(how="end")
emp_cz = emp_cz.set_index("date")
emp_cz.index = emp_cz.index.normalize()

#výběr
emp_cz = emp_cz[["OBS_VALUE"]]
emp_cz = emp_cz.rename(columns={"OBS_VALUE": "employment"})

print(emp_cz.head())
print(emp_cz.tail())


            employment
date                  
2023-06-30     5419.16
2023-09-30     5424.37
2023-12-31     5432.43
2024-03-31     5425.49
2024-06-30     5454.66
            employment
date                  
2025-03-31     5483.97
2025-06-30     5509.54
2025-09-30     5520.55
2025-12-31     5520.65
2026-03-31     5522.62


In [34]:
# growth rate
emp_cz["emp_growth"] = emp_cz["employment"].pct_change() * 100
emp_cz = emp_cz[["emp_growth"]]

print(emp_cz.head())
print(emp_cz.tail())

            emp_growth
date                  
2023-06-30         NaN
2023-09-30    0.096140
2023-12-31    0.148589
2024-03-31   -0.127751
2024-06-30    0.537647
            emp_growth
date                  
2025-03-31    0.369892
2025-06-30    0.466268
2025-09-30    0.199835
2025-12-31    0.001811
2026-03-31    0.035684


Proměnná zaměstnanosti nebyla zahrnuta do finálního datasetu, 
protože je dostupná pouze pro omezené období (od roku 2023), 
což by vedlo k výraznému zkrácení časové řady.

PREPROCESSING

Data byla upravena tak, aby byla vhodná pro modelování.

Byly provedeny následující kroky:
- převod dat na kvartální frekvenci
- výpočet meziroční inflace (YoY)
- výpočet růstu HDP
- sjednocení časového indexu
- odstranění chybějících hodnot
- odstranění duplicit

Všechny kroky byly navrženy tak, aby nevznikal data leakage.

DATASET

In [35]:
# sjednocení indexů na čisté datum bez času
infl_cz_q.index = infl_cz_q.index.normalize()
infl_de_q.index = infl_de_q.index.normalize()
rate_q.index = rate_q.index.normalize()
oil_q.index = oil_q.index.normalize()
gdp_cz.index = gdp_cz.index.normalize()

print(infl_cz_q.index[:3])
print(infl_de_q.index[:3])
print(rate_q.index[:3])
print(oil_q.index[:3])
print(gdp_cz.index[:3])

DatetimeIndex(['1996-03-31', '1996-06-30', '1996-09-30'], dtype='datetime64[ns]', name='date', freq='QE-DEC')
DatetimeIndex(['1996-03-31', '1996-06-30', '1996-09-30'], dtype='datetime64[ns]', name='date', freq='QE-DEC')
DatetimeIndex(['1993-03-31', '1993-06-30', '1993-09-30'], dtype='datetime64[ns]', name='date', freq='QE-DEC')
DatetimeIndex(['1987-06-30', '1987-09-30', '1987-12-31'], dtype='datetime64[ns]', name='date', freq='QE-DEC')
DatetimeIndex(['1996-03-31', '1996-06-30', '1996-09-30'], dtype='datetime64[ns]', name='date', freq=None)


In [36]:

#sloučení
df = infl_cz_q.join(infl_de_q, how="inner")
df = df.join(rate_q, how="inner")
df = df.join(oil_q, how="inner")
df = df.join(gdp_cz, how="inner")

df = df.dropna()
df = df[~df.index.duplicated(keep="first")]

print(df.index.duplicated().sum())

print(df.head())
print(df.tail())

0
              infl_cz   infl_de       rate        oil  gdp_growth
date                                                             
1997-03-31   6.923669  1.590829  12.383470  21.218500   -8.675202
1997-06-30   6.271031  1.364923  19.670382  18.045156    6.774390
1997-09-30   9.243244  1.580334  15.476838  18.506230    0.070889
1997-12-31   9.595555  1.494587  16.463697  18.762381   -0.063139
1998-03-31  12.295729  0.565313  16.011698  14.104444   -7.800293
             infl_cz   infl_de      rate        oil  gdp_growth
date                                                           
2024-12-31  3.132911  2.525796  3.994726  74.656094    1.074974
2025-03-31  2.790329  2.569026  3.784599  75.874603   -5.946946
2025-06-30  2.260827  2.092048  3.575763  68.067049    7.313831
2025-09-30  2.293271  2.105469  3.495499  69.031231    0.811683
2025-12-31  1.966685  2.284426  3.539343  63.654219    0.931255


LEAKAGE AUDTI

Byla provedena kontrola možného data leakage.

Všechny vstupní proměnné odpovídají informacím dostupným v čase t.
Cílová proměnná je definována jako inflace v čase t+1.

Při konstrukci proměnných nebyly použity žádné informace z budoucnosti.

TARGET

In [37]:
df["target"] = df["infl_cz"].shift(-1)

BASELINE
Baseline = inflace z předchozího období

In [38]:
df["baseline"] = df["infl_cz"].shift(1)

In [39]:
df = df.dropna()

print(df.shape)
df.head()

(114, 7)


,infl_cz,infl_de,rate,oil,gdp_growth,target,baseline
date,,,,,,,
1997-06-30,6.271031,1.364923,19.670382,18.045156,6.774390,9.243244,6.923669
1997-09-30,9.243244,1.580334,15.476838,18.506230,0.070889,9.595555,6.271031
1997-12-31,9.595555,1.494587,16.463697,18.762381,-0.063139,12.295729,9.243244
1998-03-31,12.295729,0.565313,16.011698,14.104444,-7.800293,11.644553,9.595555
1998-06-30,11.644553,0.868908,15.611397,13.310984,7.621212,8.669770,12.295729


SPLIT

Data byla rozdělena pomocí časového splitu na trénovací, validační a testovací sadu.

Trénovací sada obsahuje historická data, validační sada slouží pro porovnání modelů
a testovací sada je vyhrazena pro finální vyhodnocení.

Trénovací sada obsahuje historická data do roku 2015, 
validační sada obsahuje novější období od roku 2016 do 2020,
testovací sada obsahuje nejnovější období 2021 - 2025


In [40]:
train = df[df.index < "2016-01-01"]
valid = df[(df.index >= "2016-01-01") & (df.index < "2021-01-01")]
test = df[df.index >= "2021-01-01"]


print(train.tail())

print(valid.head())
print(valid.tail())

print(test.head())

             infl_cz   infl_de      rate        oil  gdp_growth    target  \
date                                                                        
2014-12-31  0.471109  0.438906  0.343509  76.429219    1.750100 -0.000034   
2015-03-31 -0.000034 -0.135634  0.327778  53.977097   -5.602857  0.701570   
2015-06-30  0.701570  1.243622  0.310000  61.651111    7.811999  0.333534   
2015-09-30  0.333534  1.070313  0.305945  50.444923    1.315372 -0.000134   
2015-12-31 -0.000134  0.537025  0.289524  43.556923    1.095772  0.435041   

            baseline  
date                  
2014-12-31  0.671843  
2015-03-31  0.471109  
2015-06-30 -0.000034  
2015-09-30  0.701570  
2015-12-31  0.333534  
             infl_cz   infl_de      rate        oil  gdp_growth    target  \
date                                                                        
2016-03-31  0.435041  0.135679  0.286349  33.842742   -6.628214  0.133232   
2016-06-30  0.133232 -0.099767  0.290000  45.566875    6.932899  0.5

Data byla rozdělena pomocí časového splitu na trénovací, validační a testovací sadu.


In [41]:
# Uložení datasetů
train.to_csv("data/train.csv")
valid.to_csv("data/validation.csv")
test.to_csv("data/test.csv")

Byla vytvořena kompletní datová pipeline pro predikci inflace.

Klíčové kroky zahrnovaly:
- správnou přípravu časových řad
- eliminaci data leakage
- časový split dat
- vytvoření baseline modelu

Důležitým zjištěním bylo, že správná práce s časem a strukturou dat
je zásadní pro korektní modelování.


Všechny transformační kroky (např. výpočet inflace, agregace na kvartální úroveň)
byly deterministické a nevyžadovaly fitování na datech.

Nebyly použity žádné transformační techniky, které by využívaly statistiky z celého datasetu
(např. normalizace, škálování).

Tím bylo zajištěno, že nedochází k úniku informací z validační nebo testovací sady.